In [1]:
from pathlib import Path
import pandas as pd

print("02.20 STEP 1 - INVALID DATA INSPECTION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

SALES_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

sales_files = sorted(SALES_DIR.glob("sales_*.csv"))

print("\nSales Data")
print("-" * 65)
print("Sales files found:", len(sales_files))

total_rows = 0

for file in sales_files:
    df = pd.read_csv(file)
    total_rows += len(df)

print("Total sales rows:", total_rows)

# ---------------------------------------------------------
# Inspect first sales file
# ---------------------------------------------------------

sample_file = sales_files[0]
sample_df = pd.read_csv(sample_file)

print("\nSample File")
print("-" * 65)
print("File:", sample_file.name)

print("\nColumns")
print("-" * 65)

for column in sample_df.columns:
    print("-", column)

# ---------------------------------------------------------
# Basic data checks
# ---------------------------------------------------------

print("\nBasic Data Checks")
print("-" * 65)

print(
    "Quantity minimum:",
    sample_df["quantity"].min()
)

print(
    "Quantity maximum:",
    sample_df["quantity"].max()
)

print(
    "Discount minimum:",
    sample_df["discount_pct"].min()
)

print(
    "Discount maximum:",
    sample_df["discount_pct"].max()
)

print(
    "Unit selling price minimum:",
    sample_df["unit_selling_price"].min()
)

print(
    "Unit selling price maximum:",
    sample_df["unit_selling_price"].max()
)

# ---------------------------------------------------------
# Date validation
# ---------------------------------------------------------

sample_df["order_date"] = pd.to_datetime(
    sample_df["order_date"],
    errors="coerce"
)

print(
    "Invalid order dates:",
    sample_df["order_date"].isna().sum()
)

# ---------------------------------------------------------
# Identifier checks
# ---------------------------------------------------------

print(
    "Missing sales_line_id:",
    sample_df["sales_line_id"].isna().sum()
)

print(
    "Missing order_id:",
    sample_df["order_id"].isna().sum()
)

print(
    "Missing product_id:",
    sample_df["product_id"].isna().sum()
)

print(
    "Missing store_id:",
    sample_df["store_id"].isna().sum()
)

print("\n02.20 Step 1 inspection: COMPLETED")

02.20 STEP 1 - INVALID DATA INSPECTION

Sales Data
-----------------------------------------------------------------
Sales files found: 32
Total sales rows: 2118200

Sample File
-----------------------------------------------------------------
File: sales_2024-01.csv

Columns
-----------------------------------------------------------------
- sales_line_id
- order_id
- order_date
- order_time
- customer_id
- product_id
- store_id
- salesperson_id
- promotion_id
- sales_channel
- quantity
- unit_selling_price
- discount_pct
- payment_method
- order_status

Basic Data Checks
-----------------------------------------------------------------
Quantity minimum: 1
Quantity maximum: 4
Discount minimum: 0.0
Discount maximum: 0.0
Unit selling price minimum: 187.21
Unit selling price maximum: 68144.4
Invalid order dates: 0
Missing sales_line_id: 0
Missing order_id: 0
Missing product_id: 0
Missing store_id: 0

02.20 Step 1 inspection: COMPLETED


In [2]:
from pathlib import Path

print("02.20 STEP 2 - DEFINE INVALID DATA SCENARIO")
print("=" * 65)

BASE_DIR = Path.cwd().parent

TOTAL_SALES_ROWS = 2_118_200

# ---------------------------------------------------------
# Frozen invalid-data simulation rates
# ---------------------------------------------------------

INVALID_RATES = {
    "invalid_foreign_key": 0.0020,
    "invalid_date": 0.0008,
    "invalid_quantity": 0.0005,
    "invalid_discount": 0.0003,
    "invalid_price": 0.0003,
    "invalid_allowed_value": 0.0010,
}

# ---------------------------------------------------------
# Calculate target counts
# ---------------------------------------------------------

TARGET_COUNTS = {
    scenario: round(TOTAL_SALES_ROWS * rate)
    for scenario, rate in INVALID_RATES.items()
}

print("\nInvalid Data Scenario")
print("-" * 65)

print("Canonical sales rows:", TOTAL_SALES_ROWS)

for scenario, rate in INVALID_RATES.items():
    print(
        f"{scenario}: "
        f"{rate * 100:.2f}% "
        f"-> {TARGET_COUNTS[scenario]:,} rows"
    )

# ---------------------------------------------------------
# Invalid values to be introduced
# ---------------------------------------------------------

INVALID_VALUE_RULES = {
    "invalid_foreign_key": [
        "unknown customer_id",
        "unknown product_id",
        "unknown store_id",
        "unknown salesperson_id",
        "unknown promotion_id",
    ],
    
    "invalid_date": [
        "2026-02-30",
        "31/04/2026",
        "malformed date text",
        "date outside monthly file boundary",
    ],
    
    "invalid_quantity": [
        0,
        -1,
        -2,
    ],
    
    "invalid_discount": [
        -5,
        101,
        120,
    ],
    
    "invalid_price": [
        0,
        -100,
    ],
    
    "invalid_allowed_value": [
        "UNKNOWN_PAYMENT",
        "UNKNOWN_CHANNEL",
        "UNKNOWN_STATUS",
    ],
}

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

rates_valid = all(
    0 < rate < 1
    for rate in INVALID_RATES.values()
)

counts_valid = all(
    count > 0
    for count in TARGET_COUNTS.values()
)

rules_defined = all(
    scenario in INVALID_VALUE_RULES
    for scenario in INVALID_RATES
)

print("\nValidation")
print("-" * 65)

print("Invalid rates valid:", rates_valid)
print("Target counts valid:", counts_valid)
print("Invalid rules defined:", rules_defined)

if rates_valid and counts_valid and rules_defined:
    print("\n02.20 Step 2 validation: PASSED")
else:
    print("\n02.20 Step 2 validation: FAILED")

02.20 STEP 2 - DEFINE INVALID DATA SCENARIO

Invalid Data Scenario
-----------------------------------------------------------------
Canonical sales rows: 2118200
invalid_foreign_key: 0.20% -> 4,236 rows
invalid_date: 0.08% -> 1,695 rows
invalid_quantity: 0.05% -> 1,059 rows
invalid_discount: 0.03% -> 635 rows
invalid_price: 0.03% -> 635 rows
invalid_allowed_value: 0.10% -> 2,118 rows

Validation
-----------------------------------------------------------------
Invalid rates valid: True
Target counts valid: True
Invalid rules defined: True

02.20 Step 2 validation: PASSED


In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

print("02.20 STEP 3 - INJECT CONTROLLED INVALID DATA")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "invalid_data_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

print("Sales files found:", len(sales_files))

if len(sales_files) != 32:
    raise ValueError(
        f"Expected 32 sales files, found {len(sales_files)}"
    )

# ---------------------------------------------------------
# Approved invalid-data rates
# ---------------------------------------------------------

INVALID_RATES = {
    "invalid_foreign_key": 0.0020,
    "invalid_date": 0.0008,
    "invalid_quantity": 0.0005,
    "invalid_discount": 0.0003,
    "invalid_price": 0.0003,
    "invalid_allowed_value": 0.0010,
}

# ---------------------------------------------------------
# Calculate total rows
# ---------------------------------------------------------

total_rows = sum(
    len(pd.read_csv(file))
    for file in sales_files
)

print("Total canonical sales rows:", total_rows)

# ---------------------------------------------------------
# Target counts
# ---------------------------------------------------------

target_counts = {
    scenario: round(total_rows * rate)
    for scenario, rate in INVALID_RATES.items()
}

print("\nGlobal Injection Targets")

for scenario, count in target_counts.items():
    print(f"{scenario}: {count:,}")

# ---------------------------------------------------------
# Create non-overlapping row selections
# ---------------------------------------------------------

rng = np.random.default_rng(20260831)

global_indices = np.arange(total_rows)

available_indices = global_indices.copy()

scenario_indices = {}

for scenario, count in target_counts.items():

    selected = rng.choice(
        available_indices,
        size=count,
        replace=False
    )

    scenario_indices[scenario] = selected

    available_indices = np.setdiff1d(
        available_indices,
        selected,
        assume_unique=True
    )

# ---------------------------------------------------------
# Injection counters
# ---------------------------------------------------------

injection_summary = {
    scenario: 0
    for scenario in INVALID_RATES
}

current_position = 0

# ---------------------------------------------------------
# Process sales files
# ---------------------------------------------------------

for file in sales_files:

    df = pd.read_csv(file)

    rows_in_file = len(df)

    file_start = current_position
    file_end = current_position + rows_in_file

    file_positions = np.arange(
        file_start,
        file_end
    )

    # -----------------------------------------------------
    # IMPORTANT:
    # Keep all foreign-key columns as strings
    # -----------------------------------------------------

    foreign_key_columns = [
        "customer_id",
        "product_id",
        "store_id",
        "salesperson_id",
        "promotion_id"
    ]

    for column in foreign_key_columns:

        df[column] = df[column].astype("string")

    # =====================================================
    # 1. INVALID FOREIGN KEYS
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_foreign_key"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        choice = global_pos % 5

        if choice == 0:

            df.loc[
                local_pos,
                "customer_id"
            ] = "INVALID_CUSTOMER"

        elif choice == 1:

            df.loc[
                local_pos,
                "product_id"
            ] = "INVALID_PRODUCT"

        elif choice == 2:

            df.loc[
                local_pos,
                "store_id"
            ] = "INVALID_STORE"

        elif choice == 3:

            df.loc[
                local_pos,
                "salesperson_id"
            ] = "INVALID_SALESPERSON"

        else:

            df.loc[
                local_pos,
                "promotion_id"
            ] = "INVALID_PROMOTION"

        injection_summary["invalid_foreign_key"] += 1

    # =====================================================
    # 2. INVALID DATE
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_date"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        invalid_dates = [
            "2026-02-30",
            "31/04/2026",
            "INVALID_DATE",
            "2099-12-31"
        ]

        df.loc[
            local_pos,
            "order_date"
        ] = invalid_dates[
            global_pos % len(invalid_dates)
        ]

        injection_summary["invalid_date"] += 1

    # =====================================================
    # 3. INVALID QUANTITY
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_quantity"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        df.loc[
            local_pos,
            "quantity"
        ] = 0 if global_pos % 2 == 0 else -1

        injection_summary["invalid_quantity"] += 1

    # =====================================================
    # 4. INVALID DISCOUNT
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_discount"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        df.loc[
            local_pos,
            "discount_pct"
        ] = -5 if global_pos % 2 == 0 else 101

        injection_summary["invalid_discount"] += 1

    # =====================================================
    # 5. INVALID PRICE
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_price"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        df.loc[
            local_pos,
            "unit_selling_price"
        ] = 0 if global_pos % 2 == 0 else -100

        injection_summary["invalid_price"] += 1

    # =====================================================
    # 6. INVALID ALLOWED VALUES
    # =====================================================

    selected = np.intersect1d(
        scenario_indices["invalid_allowed_value"],
        file_positions
    )

    for global_pos in selected:

        local_pos = global_pos - file_start

        choice = global_pos % 3

        if choice == 0:

            df.loc[
                local_pos,
                "payment_method"
            ] = "UNKNOWN_PAYMENT"

        elif choice == 1:

            df.loc[
                local_pos,
                "sales_channel"
            ] = "UNKNOWN_CHANNEL"

        else:

            df.loc[
                local_pos,
                "order_status"
            ] = "UNKNOWN_STATUS"

        injection_summary["invalid_allowed_value"] += 1

    # -----------------------------------------------------
    # Save adjusted file
    # -----------------------------------------------------

    output_file = OUTPUT_DIR / file.name

    df.to_csv(
        output_file,
        index=False
    )

    current_position = file_end

# ---------------------------------------------------------
# Final summary
# ---------------------------------------------------------

print("\nInjection Summary")
print("-" * 65)

for scenario, count in injection_summary.items():

    expected = target_counts[scenario]

    print(
        f"{scenario}: "
        f"Injected={count:,} | "
        f"Expected={expected:,}"
    )

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

files_created = len(
    list(OUTPUT_DIR.glob("sales_*.csv"))
)

print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nFiles created:", files_created)

if (
    files_created == 32
    and all(
        injection_summary[s] == target_counts[s]
        for s in target_counts
    )
):

    print("\n02.20 Step 3 injection: COMPLETED")

else:

    print("\n02.20 Step 3 injection: FAILED")

02.20 STEP 3 - INJECT CONTROLLED INVALID DATA
Sales files found: 32
Total canonical sales rows: 2118200

Global Injection Targets
invalid_foreign_key: 4,236
invalid_date: 1,695
invalid_quantity: 1,059
invalid_discount: 635
invalid_price: 635
invalid_allowed_value: 2,118

Injection Summary
-----------------------------------------------------------------
invalid_foreign_key: Injected=4,236 | Expected=4,236
invalid_date: Injected=1,695 | Expected=1,695
invalid_quantity: Injected=1,059 | Expected=1,059
invalid_discount: Injected=635 | Expected=635
invalid_price: Injected=635 | Expected=635
invalid_allowed_value: Injected=2,118 | Expected=2,118

Output folder:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\invalid_data_adjusted\sales_adjusted

Files created: 32

02.20 Step 3 injection: COMPLETED


In [6]:
from pathlib import Path
import pandas as pd

print("02.20 STEP 4 - VALIDATE INVALID DATA")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "invalid_data_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(
    OUTPUT_DIR.glob("sales_*.csv")
)

print("\nValidation Data")
print("-" * 65)

print("Sales files:", len(sales_files))

# ---------------------------------------------------------
# Expected values
# ---------------------------------------------------------

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

EXPECTED_INVALID_FOREIGN_KEY = 4_236
EXPECTED_INVALID_DATE = 1_695
EXPECTED_INVALID_QUANTITY = 1_059
EXPECTED_INVALID_DISCOUNT = 635
EXPECTED_INVALID_PRICE = 635
EXPECTED_INVALID_ALLOWED_VALUE = 2_118

# ---------------------------------------------------------
# Counters
# ---------------------------------------------------------

total_rows = 0

invalid_foreign_key = 0
invalid_date = 0
invalid_quantity = 0
invalid_discount = 0
invalid_price = 0
invalid_allowed_value = 0

# ---------------------------------------------------------
# Process files
# ---------------------------------------------------------

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # -----------------------------------------------------
    # Invalid foreign keys
    # -----------------------------------------------------

    invalid_foreign_key += (
        df["customer_id"].astype("string")
        .eq("INVALID_CUSTOMER")
        .sum()
    )

    invalid_foreign_key += (
        df["product_id"].astype("string")
        .eq("INVALID_PRODUCT")
        .sum()
    )

    invalid_foreign_key += (
        df["store_id"].astype("string")
        .eq("INVALID_STORE")
        .sum()
    )

    invalid_foreign_key += (
        df["salesperson_id"].astype("string")
        .eq("INVALID_SALESPERSON")
        .sum()
    )

    invalid_foreign_key += (
        df["promotion_id"].astype("string")
        .eq("INVALID_PROMOTION")
        .sum()
    )

    # -----------------------------------------------------
    # Invalid dates
    # -----------------------------------------------------

    invalid_dates = [
        "2026-02-30",
        "31/04/2026",
        "INVALID_DATE",
        "2099-12-31"
    ]

    invalid_date += df["order_date"].astype("string").isin(
        invalid_dates
    ).sum()

    # -----------------------------------------------------
    # Invalid quantity
    # -----------------------------------------------------

    invalid_quantity += (
        pd.to_numeric(
            df["quantity"],
            errors="coerce"
        )
        .isin([0, -1])
        .sum()
    )

    # -----------------------------------------------------
    # Invalid discount
    # -----------------------------------------------------

    invalid_discount += (
        pd.to_numeric(
            df["discount_pct"],
            errors="coerce"
        )
        .isin([-5, 101])
        .sum()
    )

    # -----------------------------------------------------
    # Invalid price
    # -----------------------------------------------------

    invalid_price += (
        pd.to_numeric(
            df["unit_selling_price"],
            errors="coerce"
        )
        .isin([0, -100])
        .sum()
    )

    # -----------------------------------------------------
    # Invalid allowed values
    # -----------------------------------------------------

    invalid_allowed_value += (
        df["payment_method"]
        .astype("string")
        .eq("UNKNOWN_PAYMENT")
        .sum()
    )

    invalid_allowed_value += (
        df["sales_channel"]
        .astype("string")
        .eq("UNKNOWN_CHANNEL")
        .sum()
    )

    invalid_allowed_value += (
        df["order_status"]
        .astype("string")
        .eq("UNKNOWN_STATUS")
        .sum()
    )

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("\nValidation Results")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Total rows:", total_rows)

print("\nInvalid scenarios detected:")

print(
    "Invalid foreign keys:",
    invalid_foreign_key,
    "| Expected:",
    EXPECTED_INVALID_FOREIGN_KEY
)

print(
    "Invalid dates:",
    invalid_date,
    "| Expected:",
    EXPECTED_INVALID_DATE
)

print(
    "Invalid quantities:",
    invalid_quantity,
    "| Expected:",
    EXPECTED_INVALID_QUANTITY
)

print(
    "Invalid discounts:",
    invalid_discount,
    "| Expected:",
    EXPECTED_INVALID_DISCOUNT
)

print(
    "Invalid prices:",
    invalid_price,
    "| Expected:",
    EXPECTED_INVALID_PRICE
)

print(
    "Invalid allowed values:",
    invalid_allowed_value,
    "| Expected:",
    EXPECTED_INVALID_ALLOWED_VALUE
)

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

validation_passed = (
    len(sales_files) == EXPECTED_FILES
    and total_rows == EXPECTED_ROWS
    and invalid_foreign_key == EXPECTED_INVALID_FOREIGN_KEY
    and invalid_date == EXPECTED_INVALID_DATE
    and invalid_quantity == EXPECTED_INVALID_QUANTITY
    and invalid_discount == EXPECTED_INVALID_DISCOUNT
    and invalid_price == EXPECTED_INVALID_PRICE
    and invalid_allowed_value == EXPECTED_INVALID_ALLOWED_VALUE
)

if validation_passed:

    print("\n02.20 Step 4 validation: PASSED")

else:

    print("\n02.20 Step 4 validation: FAILED")

02.20 STEP 4 - VALIDATE INVALID DATA

Validation Data
-----------------------------------------------------------------
Sales files: 32

Validation Results
-----------------------------------------------------------------
Sales files: 32
Total rows: 2118200

Invalid scenarios detected:
Invalid foreign keys: 4236 | Expected: 4236
Invalid dates: 1695 | Expected: 1695
Invalid quantities: 1059 | Expected: 1059
Invalid discounts: 635 | Expected: 635
Invalid prices: 635 | Expected: 635
Invalid allowed values: 2118 | Expected: 2118

02.20 Step 4 validation: PASSED


In [7]:
from pathlib import Path
import pandas as pd

print("02.20 STEP 5 - FINAL INVALID DATA VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "invalid_data_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(
    OUTPUT_DIR.glob("sales_*.csv")
)

EXPECTED_FILES = 32
EXPECTED_ROWS = 2_118_200

EXPECTED_COUNTS = {
    "invalid_foreign_key": 4_236,
    "invalid_date": 1_695,
    "invalid_quantity": 1_059,
    "invalid_discount": 635,
    "invalid_price": 635,
    "invalid_allowed_value": 2_118
}

total_rows = 0

invalid_foreign_key = 0
invalid_date = 0
invalid_quantity = 0
invalid_discount = 0
invalid_price = 0
invalid_allowed_value = 0

# ---------------------------------------------------------
# Validate every adjusted file
# ---------------------------------------------------------

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Foreign keys
    invalid_foreign_key += (
        df["customer_id"].astype("string")
        .eq("INVALID_CUSTOMER")
        .sum()
    )

    invalid_foreign_key += (
        df["product_id"].astype("string")
        .eq("INVALID_PRODUCT")
        .sum()
    )

    invalid_foreign_key += (
        df["store_id"].astype("string")
        .eq("INVALID_STORE")
        .sum()
    )

    invalid_foreign_key += (
        df["salesperson_id"].astype("string")
        .eq("INVALID_SALESPERSON")
        .sum()
    )

    invalid_foreign_key += (
        df["promotion_id"].astype("string")
        .eq("INVALID_PROMOTION")
        .sum()
    )

    # Invalid dates
    invalid_date += (
        df["order_date"]
        .astype("string")
        .isin([
            "2026-02-30",
            "31/04/2026",
            "INVALID_DATE",
            "2099-12-31"
        ])
        .sum()
    )

    # Invalid quantity
    invalid_quantity += (
        pd.to_numeric(
            df["quantity"],
            errors="coerce"
        )
        .isin([0, -1])
        .sum()
    )

    # Invalid discount
    invalid_discount += (
        pd.to_numeric(
            df["discount_pct"],
            errors="coerce"
        )
        .isin([-5, 101])
        .sum()
    )

    # Invalid price
    invalid_price += (
        pd.to_numeric(
            df["unit_selling_price"],
            errors="coerce"
        )
        .isin([0, -100])
        .sum()
    )

    # Invalid allowed values
    invalid_allowed_value += (
        df["payment_method"]
        .astype("string")
        .eq("UNKNOWN_PAYMENT")
        .sum()
    )

    invalid_allowed_value += (
        df["sales_channel"]
        .astype("string")
        .eq("UNKNOWN_CHANNEL")
        .sum()
    )

    invalid_allowed_value += (
        df["order_status"]
        .astype("string")
        .eq("UNKNOWN_STATUS")
        .sum()
    )

# ---------------------------------------------------------
# Final results
# ---------------------------------------------------------

actual_counts = {
    "invalid_foreign_key": invalid_foreign_key,
    "invalid_date": invalid_date,
    "invalid_quantity": invalid_quantity,
    "invalid_discount": invalid_discount,
    "invalid_price": invalid_price,
    "invalid_allowed_value": invalid_allowed_value
}

print("\nFinal Validation Results")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Expected files:", EXPECTED_FILES)

print("\nTotal rows:", total_rows)
print("Expected rows:", EXPECTED_ROWS)

print("\nInvalid-data scenarios:")

for scenario in EXPECTED_COUNTS:

    print(
        f"{scenario}: "
        f"Actual={actual_counts[scenario]:,} | "
        f"Expected={EXPECTED_COUNTS[scenario]:,}"
    )

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

validation_passed = (
    len(sales_files) == EXPECTED_FILES
    and total_rows == EXPECTED_ROWS
    and actual_counts == EXPECTED_COUNTS
)

print("\nFinal Validation")
print("-" * 65)

if validation_passed:

    print("File count validation: PASSED")
    print("Total row validation: PASSED")
    print("Invalid scenario counts: PASSED")
    print("\n02.20 Step 5 validation: PASSED")

else:

    print("02.20 Step 5 validation: FAILED")

02.20 STEP 5 - FINAL INVALID DATA VALIDATION

Final Validation Results
-----------------------------------------------------------------
Sales files: 32
Expected files: 32

Total rows: 2118200
Expected rows: 2118200

Invalid-data scenarios:
invalid_foreign_key: Actual=4,236 | Expected=4,236
invalid_date: Actual=1,695 | Expected=1,695
invalid_quantity: Actual=1,059 | Expected=1,059
invalid_discount: Actual=635 | Expected=635
invalid_price: Actual=635 | Expected=635
invalid_allowed_value: Actual=2,118 | Expected=2,118

Final Validation
-----------------------------------------------------------------
File count validation: PASSED
Total row validation: PASSED
Invalid scenario counts: PASSED

02.20 Step 5 validation: PASSED
